# Real-Time Forecasting Dashboard

This notebook implements a real-time forecasting dashboard with:
- Live data streaming and visualization
- Real-time model predictions
- Interactive controls and model selection
- Performance monitoring and alerts
- Adaptive model updating

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, State, callback_context
import dash_bootstrap_components as dbc
from datetime import datetime, timedelta
import time
import threading
import queue
from collections import deque
import warnings

warnings.filterwarnings("ignore")

# Import forecasting models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import pickle
import json

## 1. Data Stream Simulator

In [ ]:
class DataStreamSimulator:
    """
    Simulates real-time data streaming for time series.
    """

    def __init__(
        self,
        base_value=100,
        trend=0.1,
        seasonality_period=24,
        noise_level=5,
        anomaly_prob=0.02,
    ):
        """
        Initialize the data stream simulator.

        Parameters:
        -----------
        base_value : float
            Base value of the time series
        trend : float
            Trend component
        seasonality_period : int
            Period of seasonal component
        noise_level : float
            Standard deviation of noise
        anomaly_prob : float
            Probability of generating an anomaly
        """
        self.base_value = base_value
        self.trend = trend
        self.seasonality_period = seasonality_period
        self.noise_level = noise_level
        self.anomaly_prob = anomaly_prob
        self.timestamp = datetime.now()
        self.step = 0

    def generate_next_value(self):
        """
        Generate the next value in the stream.

        Returns:
        --------
        dict : Contains timestamp, value, and anomaly flag
        """
        # Trend component
        trend_value = self.trend * self.step

        # Seasonal component
        seasonal_value = 10 * np.sin(2 * np.pi * self.step / self.seasonality_period)

        # Noise component
        noise = np.random.normal(0, self.noise_level)

        # Base value
        value = self.base_value + trend_value + seasonal_value + noise

        # Add occasional anomalies
        is_anomaly = False
        if np.random.random() < self.anomaly_prob:
            value *= np.random.choice([0.5, 1.5, 2.0])  # Sudden spike or drop
            is_anomaly = True

        # Update timestamp and step
        self.timestamp = self.timestamp + timedelta(minutes=15)  # 15-minute intervals
        self.step += 1

        return {"timestamp": self.timestamp, "value": value, "is_anomaly": is_anomaly}

    def generate_historical_data(self, n_points=500):
        """
        Generate historical data for model training.

        Parameters:
        -----------
        n_points : int
            Number of historical points to generate

        Returns:
        --------
        pd.DataFrame : Historical time series data
        """
        # Reset simulator
        self.timestamp = datetime.now() - timedelta(minutes=15 * n_points)
        self.step = 0

        data = []
        for _ in range(n_points):
            data.append(self.generate_next_value())

        return pd.DataFrame(data)

## 2. Real-Time Forecasting Engine

In [ ]:
class RealTimeForecaster:
    """
    Real-time forecasting engine with multiple models.
    """

    def __init__(self, window_size=100, forecast_horizon=10):
        """
        Initialize the forecasting engine.

        Parameters:
        -----------
        window_size : int
            Size of the sliding window for model training
        forecast_horizon : int
            Number of steps ahead to forecast
        """
        self.window_size = window_size
        self.forecast_horizon = forecast_horizon
        self.models = {}
        self.model_performance = {}
        self.scaler = StandardScaler()

    def initialize_models(self):
        """
        Initialize all forecasting models.
        """
        self.models = {
            "ARIMA": None,  # Will be fitted on demand
            "Exponential Smoothing": None,  # Will be fitted on demand
            "Random Forest": RandomForestRegressor(n_estimators=50, random_state=42),
            "Gradient Boosting": GradientBoostingRegressor(
                n_estimators=50, random_state=42
            ),
            "Simple MA": None,  # Simple moving average baseline
        }

        # Initialize performance tracking
        for model_name in self.models.keys():
            self.model_performance[model_name] = {
                "mae": deque(maxlen=50),
                "mape": deque(maxlen=50),
                "forecast_time": deque(maxlen=50),
            }

    def create_features(self, data, n_lags=10):
        """
        Create lag features for ML models.

        Parameters:
        -----------
        data : array-like
            Time series data
        n_lags : int
            Number of lag features

        Returns:
        --------
        X, y : Feature matrix and target vector
        """
        X, y = [], []
        for i in range(n_lags, len(data)):
            X.append(data[i - n_lags : i])
            y.append(data[i])
        return np.array(X), np.array(y)

    def forecast_arima(self, data):
        """
        Generate ARIMA forecast.
        """
        try:
            model = ARIMA(data, order=(2, 1, 2))
            fitted = model.fit(disp=False)
            forecast = fitted.forecast(steps=self.forecast_horizon)
            return forecast
        except:
            return np.full(self.forecast_horizon, data[-1])  # Fallback to last value

    def forecast_exp_smoothing(self, data):
        """
        Generate Exponential Smoothing forecast.
        """
        try:
            model = ExponentialSmoothing(
                data, seasonal_periods=24, seasonal="add", trend="add"
            )
            fitted = model.fit()
            forecast = fitted.forecast(steps=self.forecast_horizon)
            return forecast
        except:
            return np.full(self.forecast_horizon, data[-1])  # Fallback to last value

    def forecast_ml(self, data, model_name):
        """
        Generate ML model forecast.
        """
        try:
            # Create features
            X, y = self.create_features(data, n_lags=10)

            # Scale features
            X_scaled = self.scaler.fit_transform(X)

            # Train model
            self.models[model_name].fit(X_scaled, y)

            # Generate forecast
            last_values = data[-10:].reshape(1, -1)
            last_scaled = self.scaler.transform(last_values)

            forecast = []
            for _ in range(self.forecast_horizon):
                pred = self.models[model_name].predict(last_scaled)[0]
                forecast.append(pred)

                # Update last values for next prediction
                last_values = np.append(last_values[0, 1:], pred).reshape(1, -1)
                last_scaled = self.scaler.transform(last_values)

            return np.array(forecast)
        except:
            return np.full(self.forecast_horizon, data[-1])  # Fallback

    def forecast_simple_ma(self, data, window=10):
        """
        Simple moving average forecast.
        """
        ma = np.mean(data[-window:])
        return np.full(self.forecast_horizon, ma)

    def generate_forecasts(self, data):
        """
        Generate forecasts from all models.

        Parameters:
        -----------
        data : pd.Series or array-like
            Historical time series data

        Returns:
        --------
        dict : Forecasts from each model
        """
        forecasts = {}

        # Convert to numpy array
        if isinstance(data, pd.Series):
            data = data.values

        # Use only the most recent window
        data_window = data[-self.window_size :]

        # Generate forecasts from each model
        for model_name in self.models.keys():
            start_time = time.time()

            if model_name == "ARIMA":
                forecast = self.forecast_arima(data_window)
            elif model_name == "Exponential Smoothing":
                forecast = self.forecast_exp_smoothing(data_window)
            elif model_name in ["Random Forest", "Gradient Boosting"]:
                forecast = self.forecast_ml(data_window, model_name)
            elif model_name == "Simple MA":
                forecast = self.forecast_simple_ma(data_window)

            forecasts[model_name] = forecast

            # Track forecast time
            forecast_time = time.time() - start_time
            self.model_performance[model_name]["forecast_time"].append(forecast_time)

        return forecasts

    def update_performance(self, model_name, actual, predicted):
        """
        Update model performance metrics.

        Parameters:
        -----------
        model_name : str
            Name of the model
        actual : array-like
            Actual values
        predicted : array-like
            Predicted values
        """
        mae = np.mean(np.abs(actual - predicted))
        mape = np.mean(np.abs((actual - predicted) / actual)) * 100

        self.model_performance[model_name]["mae"].append(mae)
        self.model_performance[model_name]["mape"].append(mape)

    def get_best_model(self, metric="mae"):
        """
        Get the best performing model based on recent performance.

        Parameters:
        -----------
        metric : str
            Performance metric to use ('mae' or 'mape')

        Returns:
        --------
        str : Name of the best model
        """
        avg_performance = {}

        for model_name in self.models.keys():
            if len(self.model_performance[model_name][metric]) > 0:
                avg_performance[model_name] = np.mean(
                    self.model_performance[model_name][metric]
                )

        if avg_performance:
            return min(avg_performance, key=avg_performance.get)
        return "Simple MA"  # Default

## 3. Alert System

In [ ]:
class AlertSystem:
    """
    Alert system for monitoring forecasts and data quality.
    """

    def __init__(self):
        """
        Initialize the alert system.
        """
        self.alerts = deque(maxlen=100)
        self.thresholds = {
            "anomaly_threshold": 2.5,  # Standard deviations
            "forecast_error_threshold": 0.15,  # 15% error
            "data_quality_threshold": 0.1,  # 10% missing data
            "model_degradation_threshold": 0.2,  # 20% performance drop
        }

    def check_anomaly(self, value, historical_values):
        """
        Check if a value is anomalous.

        Parameters:
        -----------
        value : float
            Current value to check
        historical_values : array-like
            Historical values for comparison

        Returns:
        --------
        bool : True if anomalous
        """
        mean = np.mean(historical_values)
        std = np.std(historical_values)
        z_score = abs((value - mean) / std)

        if z_score > self.thresholds["anomaly_threshold"]:
            self.add_alert(
                level="WARNING",
                type="ANOMALY",
                message=f"Anomalous value detected: {value:.2f} (Z-score: {z_score:.2f})",
            )
            return True
        return False

    def check_forecast_error(self, actual, predicted, model_name):
        """
        Check if forecast error exceeds threshold.

        Parameters:
        -----------
        actual : float
            Actual value
        predicted : float
            Predicted value
        model_name : str
            Name of the model
        """
        error = abs((actual - predicted) / actual)

        if error > self.thresholds["forecast_error_threshold"]:
            self.add_alert(
                level="WARNING",
                type="FORECAST_ERROR",
                message=f"{model_name} forecast error: {error * 100:.1f}%",
            )

    def check_data_quality(self, data):
        """
        Check data quality issues.

        Parameters:
        -----------
        data : pd.DataFrame
            Data to check
        """
        missing_ratio = data.isnull().sum().sum() / data.size

        if missing_ratio > self.thresholds["data_quality_threshold"]:
            self.add_alert(
                level="ERROR",
                type="DATA_QUALITY",
                message=f"High missing data ratio: {missing_ratio * 100:.1f}%",
            )

    def check_model_degradation(
        self, current_performance, baseline_performance, model_name
    ):
        """
        Check for model performance degradation.

        Parameters:
        -----------
        current_performance : float
            Current model performance
        baseline_performance : float
            Baseline performance for comparison
        model_name : str
            Name of the model
        """
        degradation = (
            current_performance - baseline_performance
        ) / baseline_performance

        if degradation > self.thresholds["model_degradation_threshold"]:
            self.add_alert(
                level="WARNING",
                type="MODEL_DEGRADATION",
                message=f"{model_name} performance degraded by {degradation * 100:.1f}%",
            )

    def add_alert(self, level, type, message):
        """
        Add an alert to the system.

        Parameters:
        -----------
        level : str
            Alert level (INFO, WARNING, ERROR)
        type : str
            Alert type
        message : str
            Alert message
        """
        alert = {
            "timestamp": datetime.now(),
            "level": level,
            "type": type,
            "message": message,
        }
        self.alerts.append(alert)

    def get_recent_alerts(self, n=10):
        """
        Get recent alerts.

        Parameters:
        -----------
        n : int
            Number of alerts to return

        Returns:
        --------
        list : Recent alerts
        """
        return list(self.alerts)[-n:]

    def get_alert_summary(self):
        """
        Get summary of alerts by type.

        Returns:
        --------
        dict : Alert summary
        """
        summary = {"INFO": 0, "WARNING": 0, "ERROR": 0}

        for alert in self.alerts:
            summary[alert["level"]] += 1

        return summary

## 4. Dashboard Application

In [ ]:
class RealTimeForecastingDashboard:
    """
    Real-time forecasting dashboard with Dash and Plotly.
    """

    def __init__(self):
        """
        Initialize the dashboard.
        """
        # Initialize components
        self.simulator = DataStreamSimulator()
        self.forecaster = RealTimeForecaster()
        self.alert_system = AlertSystem()

        # Initialize forecaster models
        self.forecaster.initialize_models()

        # Data storage
        self.historical_data = self.simulator.generate_historical_data(500)
        self.streaming_data = deque(maxlen=1000)
        self.forecasts_history = {
            model: deque(maxlen=100) for model in self.forecaster.models.keys()
        }

        # Initialize Dash app
        self.app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

        # Setup layout and callbacks
        self.setup_layout()
        self.setup_callbacks()

    def setup_layout(self):
        """
        Setup the dashboard layout.
        """
        self.app.layout = dbc.Container(
            [
                # Header
                dbc.Row(
                    [
                        dbc.Col(
                            [
                                html.H1(
                                    "Real-Time Forecasting Dashboard",
                                    className="text-center mb-4",
                                ),
                                html.Hr(),
                            ]
                        )
                    ]
                ),
                # Control Panel
                dbc.Row(
                    [
                        dbc.Col(
                            [
                                dbc.Card(
                                    [
                                        dbc.CardHeader("Control Panel"),
                                        dbc.CardBody(
                                            [
                                                dbc.Row(
                                                    [
                                                        dbc.Col(
                                                            [
                                                                html.Label(
                                                                    "Update Interval (seconds):"
                                                                ),
                                                                dcc.Slider(
                                                                    id="update-interval-slider",
                                                                    min=1,
                                                                    max=10,
                                                                    step=1,
                                                                    value=2,
                                                                    marks={
                                                                        i: str(i)
                                                                        for i in range(
                                                                            1, 11
                                                                        )
                                                                    },
                                                                ),
                                                            ],
                                                            width=6,
                                                        ),
                                                        dbc.Col(
                                                            [
                                                                html.Label(
                                                                    "Forecast Horizon:"
                                                                ),
                                                                dcc.Slider(
                                                                    id="forecast-horizon-slider",
                                                                    min=5,
                                                                    max=30,
                                                                    step=5,
                                                                    value=10,
                                                                    marks={
                                                                        i: str(i)
                                                                        for i in range(
                                                                            5, 31, 5
                                                                        )
                                                                    },
                                                                ),
                                                            ],
                                                            width=6,
                                                        ),
                                                    ]
                                                ),
                                                html.Br(),
                                                dbc.Row(
                                                    [
                                                        dbc.Col(
                                                            [
                                                                html.Label(
                                                                    "Select Models:"
                                                                ),
                                                                dcc.Dropdown(
                                                                    id="model-selector",
                                                                    options=[
                                                                        {
                                                                            "label": model,
                                                                            "value": model,
                                                                        }
                                                                        for model in self.forecaster.models.keys()
                                                                    ],
                                                                    value=list(
                                                                        self.forecaster.models.keys()
                                                                    ),
                                                                    multi=True,
                                                                ),
                                                            ],
                                                            width=8,
                                                        ),
                                                        dbc.Col(
                                                            [
                                                                html.Br(),
                                                                dbc.Button(
                                                                    "Start Streaming",
                                                                    id="streaming-button",
                                                                    color="primary",
                                                                    className="w-100",
                                                                ),
                                                            ],
                                                            width=4,
                                                        ),
                                                    ]
                                                ),
                                            ]
                                        ),
                                    ]
                                )
                            ],
                            width=12,
                        )
                    ],
                    className="mb-4",
                ),
                # Main Charts
                dbc.Row(
                    [
                        dbc.Col(
                            [
                                dbc.Card(
                                    [
                                        dbc.CardHeader("Time Series and Forecasts"),
                                        dbc.CardBody(
                                            [
                                                dcc.Graph(
                                                    id="main-chart",
                                                    style={"height": "400px"},
                                                )
                                            ]
                                        ),
                                    ]
                                )
                            ],
                            width=12,
                        )
                    ],
                    className="mb-4",
                ),
                # Performance Metrics and Alerts
                dbc.Row(
                    [
                        dbc.Col(
                            [
                                dbc.Card(
                                    [
                                        dbc.CardHeader("Model Performance"),
                                        dbc.CardBody(
                                            [
                                                dcc.Graph(
                                                    id="performance-chart",
                                                    style={"height": "300px"},
                                                )
                                            ]
                                        ),
                                    ]
                                )
                            ],
                            width=6,
                        ),
                        dbc.Col(
                            [
                                dbc.Card(
                                    [
                                        dbc.CardHeader("System Alerts"),
                                        dbc.CardBody(
                                            [
                                                html.Div(
                                                    id="alerts-panel",
                                                    style={
                                                        "height": "300px",
                                                        "overflowY": "auto",
                                                    },
                                                )
                                            ]
                                        ),
                                    ]
                                )
                            ],
                            width=6,
                        ),
                    ],
                    className="mb-4",
                ),
                # Model Comparison
                dbc.Row(
                    [
                        dbc.Col(
                            [
                                dbc.Card(
                                    [
                                        dbc.CardHeader("Model Comparison"),
                                        dbc.CardBody(
                                            [
                                                dcc.Graph(
                                                    id="comparison-chart",
                                                    style={"height": "300px"},
                                                )
                                            ]
                                        ),
                                    ]
                                )
                            ],
                            width=12,
                        )
                    ]
                ),
                # Interval component for updates
                dcc.Interval(
                    id="interval-component",
                    interval=2000,  # in milliseconds
                    n_intervals=0,
                    disabled=True,
                ),
                # Store component for data
                dcc.Store(id="streaming-data-store"),
                dcc.Store(id="forecast-data-store"),
            ],
            fluid=True,
        )

    def setup_callbacks(self):
        """
        Setup dashboard callbacks.
        """

        @self.app.callback(
            [
                Output("interval-component", "disabled"),
                Output("interval-component", "interval"),
                Output("streaming-button", "children"),
                Output("streaming-button", "color"),
            ],
            [
                Input("streaming-button", "n_clicks"),
                Input("update-interval-slider", "value"),
            ],
            [State("interval-component", "disabled")],
        )
        def toggle_streaming(n_clicks, interval_value, is_disabled):
            """Toggle streaming on/off."""
            if n_clicks:
                new_disabled = not is_disabled
                button_text = (
                    "Stop Streaming" if new_disabled == False else "Start Streaming"
                )
                button_color = "danger" if new_disabled == False else "primary"
                return new_disabled, interval_value * 1000, button_text, button_color
            return True, interval_value * 1000, "Start Streaming", "primary"

        @self.app.callback(
            [
                Output("streaming-data-store", "data"),
                Output("forecast-data-store", "data"),
            ],
            [Input("interval-component", "n_intervals")],
            [
                State("model-selector", "value"),
                State("forecast-horizon-slider", "value"),
            ],
        )
        def update_data(n, selected_models, horizon):
            """Update streaming data and forecasts."""
            # Generate new data point
            new_point = self.simulator.generate_next_value()
            self.streaming_data.append(new_point)

            # Combine historical and streaming data
            all_data = pd.concat(
                [self.historical_data, pd.DataFrame(list(self.streaming_data))]
            )["value"].values

            # Update forecast horizon
            self.forecaster.forecast_horizon = horizon

            # Generate forecasts
            forecasts = self.forecaster.generate_forecasts(all_data)

            # Filter by selected models
            filtered_forecasts = {
                k: v for k, v in forecasts.items() if k in selected_models
            }

            # Check for anomalies
            if len(all_data) > 50:
                self.alert_system.check_anomaly(new_point["value"], all_data[-50:-1])

            return list(self.streaming_data), filtered_forecasts

        @self.app.callback(
            Output("main-chart", "figure"),
            [
                Input("streaming-data-store", "data"),
                Input("forecast-data-store", "data"),
            ],
        )
        def update_main_chart(streaming_data, forecasts):
            """Update the main time series chart."""
            fig = make_subplots(rows=1, cols=1)

            # Plot historical data
            fig.add_trace(
                go.Scatter(
                    x=self.historical_data["timestamp"],
                    y=self.historical_data["value"],
                    mode="lines",
                    name="Historical",
                    line=dict(color="gray", width=1),
                )
            )

            # Plot streaming data
            if streaming_data:
                streaming_df = pd.DataFrame(streaming_data)

                # Normal points
                normal_points = streaming_df[~streaming_df["is_anomaly"]]
                if not normal_points.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=normal_points["timestamp"],
                            y=normal_points["value"],
                            mode="lines+markers",
                            name="Live Data",
                            line=dict(color="blue", width=2),
                        )
                    )

                # Anomaly points
                anomaly_points = streaming_df[streaming_df["is_anomaly"]]
                if not anomaly_points.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=anomaly_points["timestamp"],
                            y=anomaly_points["value"],
                            mode="markers",
                            name="Anomalies",
                            marker=dict(color="red", size=10, symbol="x"),
                        )
                    )

            # Plot forecasts
            if forecasts and streaming_data:
                last_timestamp = streaming_data[-1]["timestamp"]
                forecast_times = pd.date_range(
                    start=last_timestamp + timedelta(minutes=15),
                    periods=len(next(iter(forecasts.values()))),
                    freq="15min",
                )

                colors = ["green", "orange", "purple", "brown", "pink"]
                for i, (model_name, forecast_values) in enumerate(forecasts.items()):
                    fig.add_trace(
                        go.Scatter(
                            x=forecast_times,
                            y=forecast_values,
                            mode="lines",
                            name=f"{model_name} Forecast",
                            line=dict(color=colors[i % len(colors)], dash="dash"),
                        )
                    )

            fig.update_layout(
                title="Real-Time Data and Forecasts",
                xaxis_title="Time",
                yaxis_title="Value",
                hovermode="x unified",
                showlegend=True,
                legend=dict(orientation="h", y=-0.2),
            )

            return fig

        @self.app.callback(
            Output("performance-chart", "figure"),
            [Input("interval-component", "n_intervals")],
        )
        def update_performance_chart(n):
            """Update model performance chart."""
            fig = go.Figure()

            # Get performance metrics for each model
            for model_name in self.forecaster.models.keys():
                mae_values = list(self.forecaster.model_performance[model_name]["mae"])
                if mae_values:
                    fig.add_trace(
                        go.Scatter(y=mae_values, mode="lines", name=model_name)
                    )

            fig.update_layout(
                title="Model Performance (MAE)",
                xaxis_title="Time",
                yaxis_title="Mean Absolute Error",
                showlegend=True,
                legend=dict(orientation="h", y=-0.3),
            )

            return fig

        @self.app.callback(
            Output("alerts-panel", "children"),
            [Input("interval-component", "n_intervals")],
        )
        def update_alerts(n):
            """Update alerts panel."""
            recent_alerts = self.alert_system.get_recent_alerts(10)
            alert_summary = self.alert_system.get_alert_summary()

            # Create alert badges
            badges = dbc.Row(
                [
                    dbc.Col(dbc.Badge(f"Info: {alert_summary['INFO']}", color="info")),
                    dbc.Col(
                        dbc.Badge(
                            f"Warning: {alert_summary['WARNING']}", color="warning"
                        )
                    ),
                    dbc.Col(
                        dbc.Badge(f"Error: {alert_summary['ERROR']}", color="danger")
                    ),
                ],
                className="mb-3",
            )

            # Create alert list
            alert_items = []
            for alert in reversed(recent_alerts):
                color = {"INFO": "info", "WARNING": "warning", "ERROR": "danger"}[
                    alert["level"]
                ]
                alert_items.append(
                    dbc.Alert(
                        [
                            html.Strong(
                                f"{alert['timestamp'].strftime('%H:%M:%S')} - {alert['type']}"
                            ),
                            html.Br(),
                            alert["message"],
                        ],
                        color=color,
                        className="mb-2",
                    )
                )

            return html.Div([badges] + alert_items)

        @self.app.callback(
            Output("comparison-chart", "figure"),
            [Input("interval-component", "n_intervals")],
        )
        def update_comparison_chart(n):
            """Update model comparison chart."""
            # Get average performance for each model
            avg_mae = []
            avg_mape = []
            avg_time = []
            model_names = []

            for model_name in self.forecaster.models.keys():
                mae_values = list(self.forecaster.model_performance[model_name]["mae"])
                mape_values = list(
                    self.forecaster.model_performance[model_name]["mape"]
                )
                time_values = list(
                    self.forecaster.model_performance[model_name]["forecast_time"]
                )

                if mae_values:
                    model_names.append(model_name)
                    avg_mae.append(np.mean(mae_values))
                    avg_mape.append(np.mean(mape_values) if mape_values else 0)
                    avg_time.append(
                        np.mean(time_values) * 1000 if time_values else 0
                    )  # Convert to ms

            # Create subplots
            fig = make_subplots(
                rows=1,
                cols=3,
                subplot_titles=(
                    "Mean Absolute Error",
                    "Mean Absolute % Error",
                    "Forecast Time (ms)",
                ),
            )

            # MAE comparison
            fig.add_trace(
                go.Bar(x=model_names, y=avg_mae, name="MAE", marker_color="blue"),
                row=1,
                col=1,
            )

            # MAPE comparison
            fig.add_trace(
                go.Bar(x=model_names, y=avg_mape, name="MAPE", marker_color="green"),
                row=1,
                col=2,
            )

            # Time comparison
            fig.add_trace(
                go.Bar(x=model_names, y=avg_time, name="Time", marker_color="orange"),
                row=1,
                col=3,
            )

            fig.update_layout(showlegend=False, height=300)

            return fig

    def run(self, debug=True, port=8050):
        """
        Run the dashboard application.

        Parameters:
        -----------
        debug : bool
            Whether to run in debug mode
        port : int
            Port to run the application on
        """
        self.app.run_server(debug=debug, port=port)

## 5. Run Dashboard

In [ ]:
# Initialize and run the dashboard
if __name__ == "__main__":
    print("Initializing Real-Time Forecasting Dashboard...")
    dashboard = RealTimeForecastingDashboard()

    print("\nDashboard is ready!")
    print("Open your web browser and navigate to: http://127.0.0.1:8050")
    print("\nFeatures:")
    print("- Real-time data streaming with anomaly detection")
    print("- Multiple forecasting models (ARIMA, Exponential Smoothing, ML models)")
    print("- Live model performance tracking")
    print("- Alert system for anomalies and model degradation")
    print("- Interactive controls for update interval and forecast horizon")
    print("\nPress Ctrl+C to stop the server")

    # Run the dashboard
    dashboard.run(debug=False, port=8050)

## 6. Summary

This notebook demonstrates a comprehensive real-time forecasting dashboard with:

### Features Implemented:
1. **Data Streaming**: Simulated real-time data with trend, seasonality, and anomalies
2. **Multiple Models**: ARIMA, Exponential Smoothing, Random Forest, Gradient Boosting
3. **Real-Time Forecasting**: Continuous model updates and predictions
4. **Performance Monitoring**: Track MAE, MAPE, and forecast time for each model
5. **Alert System**: Detect anomalies, forecast errors, and model degradation
6. **Interactive Dashboard**: Control update frequency, forecast horizon, and model selection
7. **Model Comparison**: Visual comparison of model performance metrics

### Key Components:
- `DataStreamSimulator`: Generates realistic streaming data
- `RealTimeForecaster`: Manages multiple forecasting models
- `AlertSystem`: Monitors data quality and model performance
- `RealTimeForecastingDashboard`: Interactive web application

### Usage:
1. Run the dashboard cell to start the server
2. Open browser to http://127.0.0.1:8050
3. Click "Start Streaming" to begin real-time updates
4. Adjust controls to modify behavior
5. Monitor alerts and model performance

This dashboard provides a production-ready framework for real-time time series forecasting and monitoring.